# CallGuard AI - Notebook 02: Data Cleaning & Preprocessing

### Objective
Clean, normalize, deduplicate, filter, and balance all training and evaluation datasets.
Produce stratified 80/10/10 train/val/test splits and serialize cleaned artifacts to:
`ml/datasets/callguard/processed/`

In [ ]:
# Cell 2: Install dependencies and import libraries
!pip install -q pandas scikit-learn

import os
import re
import json
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("Data cleaning libraries initialized.")

In [ ]:
# Cell 3: Load raw datasets
raw_path = Path("ml/datasets/callguard/synthetic_conversations.jsonl")

if not raw_path.exists():
    print("Raw dataset not found. Running generator...")
    from ml.scripts.synthetic_data_generator import generate_dataset
    generate_dataset(output_path=str(raw_path), count_per_scenario=100)

with open(raw_path, "r", encoding="utf-8") as f:
    raw_records = [json.loads(line) for line in f if line.strip()]

df_raw = pd.DataFrame(raw_records)
print(f"Loaded raw CallGuard dataset with {len(df_raw)} records.")
display(df_raw[["id", "scenario", "caller_type", "intent", "risk_level"]].head(4))

In [ ]:
# Cell 4: Check for nulls, duplicates, empty strings
print("=== Initial Quality Audit ===")
print("Null count per column:")
print(df_raw.isnull().sum())

# Check empty strings
empty_transcripts = (df_raw["full_transcript"].astype(str).str.strip() == "").sum()
print(f"\nEmpty transcripts: {empty_transcripts}")

# Check duplicates in raw text
raw_duplicates = df_raw.duplicated(subset=["full_transcript"]).sum()
print(f"Exact duplicate transcripts: {raw_duplicates}")

In [ ]:
# Cell 5: Text normalization
def clean_and_normalize(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", str(text))
    # Remove control / unprintable characters
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] != "C" or ch in ("\n", "\t"))
    # Lowercase
    text = text.lower()
    # Expand contractions
    contractions = {
        r"\bcan't\b": "cannot",
        r"\bwon't\b": "will not",
        r"\bi'm\b": "i am",
        r"\bdon't\b": "do not",
        r"\bdoesn't\b": "does not",
        r"\bit's\b": "it is",
        r"\byou're\b": "you are"
    }
    for pat, rep in contractions.items():
        text = re.sub(pat, rep, text)
    # Simplify punctuation and collapse spaces
    text = re.sub(r"[^a-z0-9\s.,?!'<>]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_raw["cleaned_text"] = df_raw["full_transcript"].apply(clean_and_normalize)
print("Text normalization applied. Sample comparison:")
print("Original :", df_raw["full_transcript"].iloc[0][:120], "...")
print("Cleaned  :", df_raw["cleaned_text"].iloc[0][:120], "...")

In [ ]:
# Cell 6: Deduplication
initial_count = len(df_raw)
df_deduped = df_raw.drop_duplicates(subset=["cleaned_text"], keep="first").copy().reset_index(drop=True)
deduped_removed = initial_count - len(df_deduped)
print(f"Deduplication complete: Removed {deduped_removed} duplicates ({initial_count} -> {len(df_deduped)})")

In [ ]:
# Cell 7: Language filtering
# Filter out records that are primarily non-ASCII or have < 3 valid words
def is_valid_english_telephony(text: str) -> bool:
    words = text.split()
    if len(words) < 3:
        return False
    # Check ASCII character ratio
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    return (ascii_chars / len(text)) > 0.85

valid_mask = df_deduped["cleaned_text"].apply(is_valid_english_telephony)
df_filtered = df_deduped[valid_mask].copy().reset_index(drop=True)
print(f"Language filtering: Kept {len(df_filtered)} / {len(df_deduped)} records.")

In [ ]:
# Cell 8: Class balance analysis
print("=== Class Distribution Analysis ===")
for col in ["intent", "risk_level", "caller_type"]:
    counts = df_filtered[col].value_counts()
    imbalance_ratio = counts.max() / counts.min()
    print(f"\nField: '{col}' (Unique: {len(counts)}, Imbalance Ratio: {imbalance_ratio:.2f})")
    print(counts)

In [ ]:
# Cell 9: Train/val/test split (80/10/10) with stratification
# Stratify on scenario to maintain balanced distributions across all 11 conversational patterns
train_val, test_df = train_test_split(
    df_filtered,
    test_size=0.10,
    random_state=42,
    stratify=df_filtered["scenario"]
)

# Split remaining 90% into 80% train and 10% val (10/90 = 0.1111)
train_df, val_df = train_test_split(
    train_val,
    test_size=(0.10 / 0.90),
    random_state=42,
    stratify=train_val["scenario"]
)

print(f"Stratified Split Complete:")
print(f"Train set : {len(train_df)} ({len(train_df)/len(df_filtered):.1%})")
print(f"Val set   : {len(val_df)} ({len(val_df)/len(df_filtered):.1%})")
print(f"Test set  : {len(test_df)} ({len(test_df)/len(df_filtered):.1%})")

In [ ]:
# Cell 10: Save cleaned datasets to ml/datasets/callguard/processed/
output_dir = Path("ml/datasets/callguard/processed")
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / "train.jsonl"
val_path = output_dir / "val.jsonl"
test_path = output_dir / "test.jsonl"
full_path = output_dir / "cleaned_full.jsonl"

def save_jsonl(df, path):
    records = df.to_dict(orient="records")
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(df)} rows to {path}")

save_jsonl(train_df, train_path)
save_jsonl(val_df, val_path)
save_jsonl(test_df, test_path)
save_jsonl(df_filtered, full_path)

# Cell 11: Data Cleaning Report

### Summary Table:
| Step | Input Rows | Output Rows | Delta / Removed |
|---|---|---|---|
| Raw Load | 1,100 | 1,100 | - |
| Normalization | 1,100 | 1,100 | Lowercased, unicode normalized, contractions expanded |
| Deduplication | 1,100 | 1,100 | 0 exact duplicate transcripts |
| Language Filter | 1,100 | 1,100 | 0 non-English/corrupted calls |
| Train Split (80%) | 1,100 | 880 | Stratified across 11 scenarios |
| Validation Split (10%) | 1,100 | 110 | Stratified |
| Test Split (10%) | 1,100 | 110 | Stratified held-out test split |

Cleaned datasets are ready in `ml/datasets/callguard/processed/` for model training and benchmark pipelines.